### Setup and test

In [30]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")

print("Path to dataset files:", path)

c:\Users\2553227\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\2553227\.cache\kagglehub\datasets\jeanmidev\smart-meters-in-london\versions\21


In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

# Set Data directory
DATA_DIR = Path("data")

# For reproducibility
RNG_SEED = 42
np.random.seed(RNG_SEED)

### Load metadata (households + ACORN + tariff)

In [25]:
# acorn_path = DATA_DIR / "acorn_details.csv"
info_path  = DATA_DIR / "informations_households.csv"

# acorn_df = pd.read_csv(acorn_path, encoding="latin1")
info_df  = pd.read_csv(info_path, encoding="latin1")

# print(acorn_df.head())
print(info_df.head())

       LCLid stdorToU    Acorn Acorn_grouped     file
0  MAC005492      ToU   ACORN-        ACORN-  block_0
1  MAC001074      ToU   ACORN-        ACORN-  block_0
2  MAC000002      Std  ACORN-A      Affluent  block_0
3  MAC003613      Std  ACORN-A      Affluent  block_0
4  MAC003597      Std  ACORN-A      Affluent  block_0


In [26]:
# meta = (
#     acorn_df.merge(info_df, on="LCLid", how="inner")
# )

meta = info_df.copy()

meta.head()
# meta["stdorToU"].value_counts()
# meta["Acorn_grouped"].value_counts()

,LCLid,stdorToU,Acorn,Acorn_grouped,file
0,MAC005492,ToU,ACORN-,ACORN-,block_0
1,MAC001074,ToU,ACORN-,ACORN-,block_0
2,MAC000002,Std,ACORN-A,Affluent,block_0
3,MAC003613,Std,ACORN-A,Affluent,block_0
4,MAC003597,Std,ACORN-A,Affluent,block_0


### Sample households (balanced by tariff & ACORN_grouped)

In [27]:
# Keep only columns you need for stratification
meta = info_df[["LCLid", "stdorToU", "Acorn_grouped"]].dropna()

# Create a combined stratum label
meta["stratum"] = (
    meta["stdorToU"].astype(str) + "_" +
    meta["Acorn_grouped"].astype(str)
)

# Decide how many households per stratum (e.g. up to 50)
n_per_stratum = 2

sampled_ids = (
    meta
    .groupby("stratum", group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), n_per_stratum),
        random_state=42
    ))
    ["LCLid"]
    .unique()
)

len(sampled_ids), sampled_ids[:10]
sampled_meta = info_df[info_df["LCLid"].isin(sampled_ids)].copy()
sampled_meta["file"].value_counts()

file
block_111    3
block_0      2
block_7      1
block_18     1
block_20     1
block_42     1
block_52     1
block_55     1
block_59     1
block_68     1
block_80     1
block_91     1
block_103    1
block_106    1
block_110    1
Name: count, dtype: int64

### Find households' halfhourly data

In [28]:
blocks_to_load = sorted(sampled_meta["file"].unique())
blocks_to_load[:10], len(blocks_to_load)

dfs = []
for block in blocks_to_load:
    path = DATA_DIR / f"halfhourly_dataset/{block}.csv"
    tmp = pd.read_csv(path)
    tmp = tmp[tmp["LCLid"].isin(sampled_ids)]
    dfs.append(tmp)

hh = pd.concat(dfs, ignore_index=True)

# Ensure proper types and ordering
hh["tstp"] = pd.to_datetime(hh["tstp"])
hh = hh.sort_values(["LCLid", "tstp"])

# Count number of hh per household
hh_counts = (
    hh.groupby("LCLid")["tstp"]
         .nunique()
         .rename("n_tstp")
)

print(hh_counts.describe())


mask_year = (hh["tstp"] >= "2013-01-01") & (hh["tstp"] < "2014-01-01")
hh = hh.loc[mask_year].reset_index(drop=True)

# Keep households with enough days (e.g. at least 200)
MIN_DAYS = 200*48
good_ids = hh_counts[hh_counts >= MIN_DAYS].index

# Restrict to good households
hh = hh[hh["LCLid"].isin(good_ids)].copy()
hh.count()

count       18.000000
mean     29898.666667
std       6108.913103
min      10780.000000
25%      29034.750000
50%      30884.500000
75%      32295.000000
max      39003.000000
Name: n_tstp, dtype: float64


LCLid             288657
tstp              288657
energy(kWh/hh)    288657
dtype: int64

### Pick target

In [10]:
# Predict hh energy
hh_use = hh[["LCLid", "tstp", "energy(kWh/hh)"]].copy()

### Joing metadata

In [11]:
# Merge metadata: tariff and Acorn are the features of interest
model_df = hh_use.merge(
    sampled_meta[["LCLid", "stdorToU", "Acorn_grouped"]],
    on="LCLid",
    how="left"
)

print(model_df.head())
print(model_df.shape)

       LCLid                tstp energy(kWh/hh) stdorToU Acorn_grouped
0  MAC000038 2013-01-01 00:00:00         0.484       Std     Adversity
1  MAC000038 2013-01-01 00:30:00         0.633       Std     Adversity
2  MAC000038 2013-01-01 01:00:00         0.555       Std     Adversity
3  MAC000038 2013-01-01 01:30:00         0.318       Std     Adversity
4  MAC000038 2013-01-01 02:00:00         0.185       Std     Adversity
(428701, 5)


### Add context

In [13]:
# Sort by household and timestamp
model_df = model_df.sort_values(["LCLid", "tstp"]).reset_index(drop=True)

# Add 1-step (30 min) and 48-step (1 day) lags per household
model_df["energy_lag1"] = (
    model_df.groupby("LCLid")["energy(kWh/hh)"].shift(1)      # 30 minutes ago
)
model_df["energy_lag48"] = (
    model_df.groupby("LCLid")["energy(kWh/hh)"].shift(48)     # same time previous day
)

# Optionally: 96-step lag (2 days), 336-step lag (1 week), etc.
# model_df["energy_lag96"] = model_df.groupby("LCLid")["energy(kWh/hh)"].shift(96)
# model_df["energy_lag336"] = model_df.groupby("LCLid")["energy(kWh/hh)"].shift(336)

# Drop rows without full lag history
model_df[["energy_lag1", "energy_lag48", "energy(kWh/hh)"]] = (
    model_df[["energy_lag1", "energy_lag48", "energy(kWh/hh)"]].replace("Null", np.nan)
)

model_df = model_df.dropna(subset=["energy_lag1", "energy_lag48", "energy(kWh/hh)"]).reset_index(drop=True)

In [14]:
weather_path = DATA_DIR / "weather_hourly_darksky.csv"
weather = pd.read_csv(weather_path)

# Parse time to datetime and round/floor to the hour (safe)
weather["time"] = pd.to_datetime(weather["time"])
weather["time_hour"] = weather["time"].dt.floor("h")

# Keep useful columns (adjust list as you like)
weather = weather[[
    "time_hour",
    "temperature",
    "apparentTemperature",
    "humidity",
    "windSpeed",
    "visibility",
    "pressure",
    "windBearing"
]]

In [15]:
# Create hour key matching weather["time_hour"]
model_df["tstp_hour"] = model_df["tstp"].dt.floor("h")

# Merge half-hourly with hourly weather
model_df = model_df.merge(
    weather,
    left_on="tstp_hour",
    right_on="time_hour",
    how="left"
)

# Drop merge helper
model_df = model_df.drop(columns=["time_hour"])

In [16]:
weather_cols = ["temperature", "apparentTemperature", "humidity", "windSpeed", "visibility", "pressure", "windBearing"]
model_df[weather_cols] = model_df[weather_cols].ffill().bfill()

# Confirm
print(model_df.isnull().sum())

LCLid                  0
tstp                   0
energy(kWh/hh)         0
stdorToU               0
Acorn_grouped          0
energy_lag1            0
energy_lag48           0
tstp_hour              0
temperature            0
apparentTemperature    0
humidity               0
windSpeed              0
visibility             0
pressure               0
windBearing            0
dtype: int64


In [17]:
# Calendar context features
model_df["dayofweek"] = model_df["tstp"].dt.dayofweek
model_df["month"]     = model_df["tstp"].dt.month
model_df["season"]    = model_df["month"].map(
    {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
)

# Target = energy_mean (no shift)
model_df = model_df.dropna(subset=["energy(kWh/hh)"])

# One-hot encode small categoricals (good for both ML and simple DL MLPs)
model_df = pd.get_dummies(
    model_df,
    columns=["stdorToU", "Acorn_grouped"],
    drop_first=True
)

print(model_df.head())
print(model_df.shape)

       LCLid                tstp energy(kWh/hh) energy_lag1 energy_lag48  \
0  MAC000038 2013-01-03 00:00:00         0.237       0.607        0.292    
1  MAC000038 2013-01-03 00:30:00         0.066       0.237        0.167    
2  MAC000038 2013-01-03 01:00:00         0.054       0.066         0.06    
3  MAC000038 2013-01-03 01:30:00         0.069       0.054        0.056    
4  MAC000038 2013-01-03 02:00:00         0.085       0.069        0.055    

            tstp_hour  temperature  apparentTemperature  humidity  windSpeed  \
0 2013-01-03 00:00:00        10.69                10.69      0.91       4.78   
1 2013-01-03 00:00:00        10.69                10.69      0.91       4.78   
2 2013-01-03 01:00:00        10.31                10.31      0.94       4.18   
3 2013-01-03 01:00:00        10.31                10.31      0.94       4.18   
4 2013-01-03 02:00:00        10.27                10.27      0.93       4.33   

   ...  pressure  windBearing  dayofweek  month  season  stdor

In [18]:
# Load UK bank holidays
hol_path = DATA_DIR / "uk_bank_holidays.csv"
hol = pd.read_csv(hol_path)

# Parse holiday dates
hol["day"] = pd.to_datetime(hol["Bank holidays"])
hol["day"] = hol["day"].dt.normalize()          # midnight
hol["is_bank_holiday"] = True

holidays = hol[["day", "is_bank_holiday"]].drop_duplicates()

# Add a date column to half-hourly data
model_df["day"] = model_df["tstp"].dt.normalize()

# Merge
model_df = model_df.merge(holidays, on="day", how="left")
model_df["is_bank_holiday"] = model_df["is_bank_holiday"].fillna(False)

### Save dataset

In [19]:
dataset_path = "smart_meters_hh_supervised.csv"
model_df.to_csv(dataset_path, index=False)

dataset_path

'smart_meters_hh_supervised.csv'

### Extra: test model for meaningful results

In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
import pandas as pd
import numpy as np
from pathlib import Path

dataset_path = "smart_meters_hh_supervised.csv"
model_df = pd.read_csv(dataset_path)
model_df["tstp"] = pd.to_datetime(model_df["tstp"])

# Target
y = model_df["energy(kWh/hh)"]

# Features (all except ID, day, target)
feature_cols = [
    "dayofweek",
    "month",
    "season",
    "stdorToU_ToU",
    "Acorn_grouped_ACORN-U",
    "Acorn_grouped_Adversity",
    "Acorn_grouped_Affluent",
    "Acorn_grouped_Comfortable",
    "temperature", 
    "apparentTemperature",
    "humidity",
    "windSpeed",
    "visibility",
    "pressure",
    "windBearing",
    "is_bank_holiday",
    "energy_lag1",
    "energy_lag48",
]

# Time-based split: last 20% of days as validation
cutoff_date = model_df["tstp"].quantile(0.8)

train_mask = model_df["tstp"] <= cutoff_date
val_mask   = model_df["tstp"] >  cutoff_date

X_train = model_df.loc[train_mask, feature_cols]
y_train = y[train_mask]

X_val   = model_df.loc[val_mask, feature_cols]
y_val   = y[val_mask]

In [21]:
lin = LinearRegression()
lin.fit(X_train, y_train)

y_pred = lin.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)
r2   = r2_score(y_val, y_pred)

rmse, r2

(0.17332967091658766, 0.69761525178513)

In [22]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lin.coef_
}).sort_values("coefficient", key=abs, ascending=False)

print(coef_df)

                      feature  coefficient
16                energy_lag1     0.745569
17               energy_lag48     0.168152
5     Acorn_grouped_Adversity    -0.010858
3                stdorToU_ToU    -0.010141
10                   humidity    -0.008054
7   Acorn_grouped_Comfortable    -0.006096
15            is_bank_holiday    -0.004923
4       Acorn_grouped_ACORN-U    -0.003710
6      Acorn_grouped_Affluent    -0.002763
2                      season     0.001133
0                   dayofweek    -0.000932
9         apparentTemperature    -0.000689
1                       month    -0.000666
8                 temperature     0.000654
12                 visibility    -0.000526
11                  windSpeed     0.000460
13                   pressure     0.000006
14                windBearing     0.000002


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
import pandas as pd
import numpy as np
from pathlib import Path
RNG_SEED = 42
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=RNG_SEED,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

rmse_rf = root_mean_squared_error(y_val, y_pred_rf)
r2_rf   = r2_score(y_val, y_pred_rf)

rmse_rf, r2_rf

(0.17542798611119756, 0.6902496449314086)

In [24]:
imp_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

print(imp_df)

                      feature  importance
16                energy_lag1    0.755414
17               energy_lag48    0.055532
13                   pressure    0.022737
11                  windSpeed    0.022049
14                windBearing    0.020473
12                 visibility    0.018418
4       Acorn_grouped_ACORN-U    0.017256
10                   humidity    0.016998
9         apparentTemperature    0.016555
8                 temperature    0.016284
0                   dayofweek    0.009727
3                stdorToU_ToU    0.007103
1                       month    0.007095
7   Acorn_grouped_Comfortable    0.005188
6      Acorn_grouped_Affluent    0.004412
2                      season    0.002351
5     Acorn_grouped_Adversity    0.001733
15            is_bank_holiday    0.000675
